## Data Preprocessing

In [32]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
df=pd.read_csv("../data/student_dataset/student-mat.csv", sep=";")
df.head(5)

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [33]:
#Seperate input features from target
target="G3"
X=df.drop(columns=[target])
y=df[target]
print(y)

0       6
1       6
2      10
3      15
4      10
       ..
390     9
391    16
392     7
393    10
394     9
Name: G3, Length: 395, dtype: int64


In [34]:
#numerical and categorical columns
numerical_cols=X.select_dtypes(include="number").columns.tolist()
categorical_cols=X.select_dtypes(include=["object", "category"]).columns.tolist()
print(numerical_cols)
print(categorical_cols)

['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


C:\Users\anamo\AppData\Local\Temp\ipykernel_23104\1927798155.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols=X.select_dtypes(include=["object", "category"]).columns.tolist()


In [35]:
##Split
X_train, X_test, y_train, y_test= train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
    )
print(f"Training Features: {X_train.shape}")
print(f"Testing Features: {X_test.shape}")
print(f"Training Targets: {y_train.shape}")
print(f"Testing Targets: {y_test.shape}")

Training Features: (316, 32)
Testing Features: (79, 32)
Training Targets: (316,)
Testing Targets: (79,)


In [36]:
##Preprocessing Transformers
numerical_transformer=StandardScaler()
categorical_transformer=OneHotEncoder()
preprocessor=ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ], remainder="drop"
)
print("Preprocessor created successfully.")

Preprocessor created successfully.


In [39]:
##Baseline regression using Ridge
model=Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0))
])
print(model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'Medu', 'Fedu',
                                                   'traveltime', 'studytime',
                                                   'failures', 'famrel',
                                                   'freetime', 'goout', 'Dalc',
                                                   'Walc', 'health', 'absences',
                                                   'G1', 'G2']),
                                                 ('cat', OneHotEncoder(),
                                                  ['school', 'sex', 'address',
                                                   'famsize', 'Pstatus', 'Mjob',
                                                   'Fjob', 'reason', 'guardian',
                                                   'schoolsup', 'famsup',
                                                   '

In [40]:
# Train Transform and evaluations
model.fit(X_train, y_train)
X_train_processed=model.named_steps["preprocessor"].transform(X_train)
X_test_processed=model.named_steps["preprocessor"].transform(X_test)
print(f"Processed Training Shape: {X_train_processed.shape} and Processed testing shape: {X_test_processed.shape}")
y_pred=model.predict(X_test)
mae= mean_absolute_error(y_test, y_pred)
rmse=mean_squared_error(y_test, y_pred)**0.5
r2= r2_score(y_test, y_pred)
print(f"Mean Absoulte Error: {mae}, Root-Mean Absoulte Error: {rmse}, R2 Scrore: {r2}")

Processed Training Shape: (316, 58) and Processed testing shape: (79, 58)
Mean Absoulte Error: 1.6390671341990937, Root-Mean Absoulte Error: 2.3715321282176913, R2 Scrore: 0.7257180360697023
